# Actividad 2 — Reconocimiento del dataset

**Curso:** Proyecto Integrado III - Analítica de Datos
**Dataset:** `t22000t/bike-sharing-tabular` (UCI Bike Sharing Dataset — Capital Bikeshare, Washington D.C., 2011–2012)
**Split:** `train`

## Propósito

Este notebook **no hace un análisis detallado** ni un análisis estadístico de la demanda.

Hace un **análisis descriptivo del conjunto de datos** (metadata descriptiva): describe qué contiene el
dataset, cómo está estructurado, qué tipos tiene cada columna, qué dominios y rangos admite y cuánta
cobertura temporal ofrece. La media, la varianza y el análisis del comportamiento de la demanda
corresponden a la etapa de métricas, no a esta entrega.

## Qué hace paso a paso:


1. Obtener el dataset desde Hugging Face con la clase `HFDatasetLoader`.
2. Mostrar registros, dimensiones, columnas, tipos y valores nulos.
3. Explicar qué representa cada fila y qué variables respaldan la necesidad.
4. Generar automáticamente el resumen del DataFrame en `AE2/resultados/resumen_dataframe.txt`.

## 1. Preparar el entorno

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader, DatasetSummarizer

print("Raíz del proyecto:", ROOT)

Raíz del proyecto: /home/runner/work/proyecto-integrado-iii/proyecto-integrado-iii


/opt/hostedtoolcache/Python/3.12.16/x64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Obtener el dataset desde Hugging Face

Se carga el dataset completo (no una muestra): 17.379 registros.

In [2]:
loader = HFDatasetLoader(
    dataset_name="t22000t/bike-sharing-tabular",
    split="train",
)

df = loader.load()

print("Dimensiones:", df.shape[0], "filas x", df.shape[1], "columnas")
df.head()

Dimensiones: 17379 filas x 17 columnas


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


## 3. Qué representa cada fila

**Una fila = una hora de operación** del sistema de bicicletas públicas de Washington D.C.

No es una persona ni una bicicleta individual: es un **registro agregado horario** que dice cuántas bicicletas se rentaron en esa hora, bajo qué condiciones climáticas y en qué momento del calendario.

Variables que respaldan la necesidad de información:

- `cnt` (total de rentas de la hora) → la **cantidad** que se quiere caracterizar
- `hr` (hora del día), `weekday` (día de la semana), `mnth`, `season`, `yr` → permiten comparar **grupos y periodos**
- `weathersit` (condición climática), `temp`, `atemp`, `hum`, `windspeed` → permiten comparar la demanda según el **contexto ambiental**
- `casual` y `registered` → permiten comparar **dos tipos de usuario**
- `workingday` y `holiday` → distinguen la **movilidad cotidiana** del uso recreativo

Con esas variables se puede comparar cantidades, categorías, periodos y grupos, que es exactamente lo que pide un proyecto de analítica descriptiva.

## 4. Dimensiones, columnas y tipos

In [3]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 17379
Columnas: 17
<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  str    
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), str(1)
memory usage: 2.4 MB


## 5. Valores nulos por columna

In [4]:
nulos = df.isna().sum()
print(nulos.to_string())
print("\nTotal de valores nulos:", int(nulos.sum()))
print("Filas duplicadas:", int(df.duplicated().sum()))

instant       0
dteday        0
season        0
yr            0
mnth          0
hr            0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0

Total de valores nulos: 0
Filas duplicadas: 0


## 6. Reconocimiento de las variables categóricas

Las variables categóricas vienen **codificadas con números**. Su significado está documentado en el README oficial del dataset (UCI, id 275), no se deduce de los datos.

In [5]:
for col in ["season", "yr", "weathersit", "holiday", "workingday", "weekday"]:
    conteo = df[col].value_counts().sort_index()
    print(f"\n--- {col} ---")
    print(conteo.to_string())


--- season ---
season
1    4242
2    4409
3    4496
4    4232

--- yr ---
yr
0    8645
1    8734

--- weathersit ---
weathersit
1    11413
2     4544
3     1419
4        3

--- holiday ---
holiday
0    16879
1      500

--- workingday ---
workingday
0     5514
1    11865

--- weekday ---
weekday
0    2502
1    2479
2    2453
3    2475
4    2471
5    2487
6    2512


## 7. Generar el resumen del DataFrame en un archivo .txt

El archivo se genera por código a partir de **este mismo DataFrame**, se escribe en UTF-8, en una ruta relativa del proyecto, y se **regenera en cada ejecución**. La clase crea la carpeta si no existe.

> El archivo incluye, además de la metadata, un apéndice con estadísticas descriptivas. Se conserva solo como referencia para reconocer los rangos de las variables; el análisis del comportamiento de la demanda corresponde a la etapa de métricas.


In [6]:
summarizer = DatasetSummarizer(
    dataset_id="t22000t/bike-sharing-tabular",
    split="train",
    nota="Dataset completo (sin muestra ni filtros).",
)

ruta = summarizer.guardar(df, ROOT / "AE2" / "resultados" / "resumen_dataframe.txt")

print("Archivo generado:", ruta.relative_to(ROOT))

Archivo generado: AE2/resultados/resumen_dataframe.txt


## 8. Verificar el contenido del archivo

In [7]:
contenido = ruta.read_text(encoding="utf-8")

# Comprobación: el archivo debe declarar el dataset y el split utilizados.
for obligatorio in ["Dataset: t22000t/bike-sharing-tabular", "Split: train", "Filas: 17379"]:
    print(("OK   " if obligatorio in contenido else "FALTA ") + obligatorio)

print()
print(contenido[:1500])

OK   Dataset: t22000t/bike-sharing-tabular
OK   Split: train
OK   Filas: 17379

RESUMEN DEL DATAFRAME
Dataset: t22000t/bike-sharing-tabular
Split: train
Generado: 2026-11-12 15:56
Nota: Dataset completo (sin muestra ni filtros).

DIMENSIONES
--------------------------------------------------------------
Filas: 17379
Columnas: 17
Memoria aproximada: 2.42 MB
Filas duplicadas: 0
Valores nulos totales: 0

COLUMNAS Y TIPOS
--------------------------------------------------------------
- instant: int64
- dteday: str
- season: int64
- yr: int64
- mnth: int64
- hr: int64
- holiday: int64
- weekday: int64
- workingday: int64
- weathersit: int64
- temp: float64
- atemp: float64
- hum: float64
- windspeed: float64
- casual: int64
- registered: int64
- cnt: int64

VALORES NULOS POR COLUMNA
--------------------------------------------------------------
- instant: 0 (0.00 %)
- dteday: 0 (0.00 %)
- season: 0 (0.00 %)
- yr: 0 (0.00 %)
- mnth: 0 (0.00 %)
- hr: 0 (0.00 %)
- holiday: 0 (0.00 %)
- weekday

## 9. Conclusión

- El dataset se obtuvo desde Hugging Face y se cargó en un `DataFrame` de pandas (17.379 × 17).
- El DataFrame no presenta valores nulos ni filas duplicadas.
- Cada fila representa una hora de operación del sistema; las variables numéricas y categóricas permiten abordar la necesidad planteada en el documento.
- El resumen automático quedó en `AE2/resultados/resumen_dataframe.txt`, generado por código a partir de este mismo DataFrame.